In [1]:
# ============================================================
# CELL 1: IMPORTS + NOTEBOOK SETTINGS
# ============================================================
# Goal of this cell:
# 1) Import the libraries we need (pandas/numpy/sklearn)
# 2) Configure the notebook to display results clearly
# 3) Reduce warning noise so outputs are easier to read

# ---- Data handling libraries ----
import pandas as pd                # pandas = tables (DataFrame), reading CSV, cleaning, analysis
import numpy as np                 # numpy = numerical operations, arrays, math, sqrt, etc.

# ---- Warning control ----
import warnings                    # used to control Python warnings
warnings.filterwarnings("ignore")  # hide warnings (not errors) to keep the notebook output clean

# ---- Scikit-learn: training/testing + evaluation ----
from sklearn.model_selection import train_test_split     # splits data into train and test subsets
from sklearn.metrics import mean_squared_error, r2_score # metrics for regression evaluation

# ---- Scaling (important for SVR) ----
from sklearn.preprocessing import StandardScaler         # standardize features: mean=0, std=1

# ---- DataFrame display settings ----
pd.set_option("display.max_columns", None)  # show all columns when printing DataFrames
pd.set_option("display.width", 150)         # widen display to avoid line wrapping


In [2]:
# ============================================================
# CELL 2: DATA LOADING
# ============================================================
# Goal of this cell:
# - Read the NBA CSV dataset into a pandas DataFrame named df
# - Print shape to confirm loading success
# - Display first rows to verify columns/data look correct

data_path = "database_24_25.csv"   # the CSV file name/path (same folder as notebook)
df = pd.read_csv(data_path)        # read CSV into a DataFrame (table-like structure)

print("✅ Data loaded successfully from CSV!")  # confirmation message
print("Dataset shape (rows, columns):", df.shape)  # show number of rows and columns

df.head()  # display first 5 rows (quick sanity check)


✅ Data loaded successfully from CSV!
Dataset shape (rows, columns): (16512, 25)


,Player,Tm,Opp,Res,MP,FG,FGA,FG%,3P,3PA,3P%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,GmSc,Data
0,Jayson Tatum,BOS,NYK,W,30.30,14,18,0.778,8,11,0.727,1,2,0.500,0,4,4,10,1,1,1,1,37,38.1,2024-10-22
1,Anthony Davis,LAL,MIN,W,37.58,11,23,0.478,1,3,0.333,13,15,0.867,3,13,16,4,1,3,1,1,36,34.0,2024-10-22
2,Derrick White,BOS,NYK,W,26.63,8,13,0.615,6,10,0.600,2,2,1.000,0,3,3,4,1,0,0,1,24,22.4,2024-10-22
3,Jrue Holiday,BOS,NYK,W,30.52,7,9,0.778,4,6,0.667,0,0,0.000,2,2,4,4,1,0,0,2,18,19.5,2024-10-22
4,Miles McBride,NYK,BOS,L,25.85,8,10,0.800,4,5,0.800,2,3,0.667,0,0,0,2,0,0,1,1,22,17.8,2024-10-22


In [3]:
# ============================================================
# CELL 3: BASIC CLEANING PATTERNS
# ============================================================
# Goal of this cell:
# 1) Check missing values (how many NaNs in each column)
# 2) Remove games where the player did not play (MP <= 0)
# 3) Drop rows with missing values in key numeric columns
# 4) Remove duplicate rows
# 5) Fix MP datatype (ensure it's numeric/float)

# ---------- 1) Missing values summary ----------
missing_counts = df.isnull().sum()                 # count NaNs per column
missing_percentages = (missing_counts / len(df)) * 100  # percentage of NaNs per column

missing_summary = pd.DataFrame({
    "Missing_Count": missing_counts,               # create column for missing counts
    "Missing_Percentage": missing_percentages      # create column for missing percentage
}).sort_values("Missing_Count", ascending=False)   # sort so largest missing columns appear first

print("📋 Missing values summary (non-zero only):")
display(missing_summary[missing_summary["Missing_Count"] > 0])  # show only columns with >0 missing

# ---------- Create a clean copy so we don’t overwrite original df ----------
df_clean = df.copy()  # copy the dataset so original df stays unchanged

# ---------- 2) Remove rows where player did not play ----------
# MP = minutes played. If MP is 0 or negative, the player basically did not contribute.
if "MP" in df_clean.columns:       # check if MP column exists
    before_rows = len(df_clean)    # rows before filtering
    df_clean = df_clean[df_clean["MP"] > 0]  # keep only rows where MP > 0
    after_rows = len(df_clean)     # rows after filtering
    print(f"Filtered MP > 0: {before_rows} → {after_rows} rows")

# ---------- 3) Drop rows with missing values in key columns ----------
# These columns are important for modeling and evaluation
key_numeric_cols = ["PTS", "AST", "TRB", "FGA", "FG", "FTA", "TOV"]

# Keep only the columns that exist in the dataset (defensive coding)
existing_numeric_cols = [c for c in key_numeric_cols if c in df_clean.columns]

before_rows = len(df_clean)  # rows before dropping NaNs
df_clean = df_clean.dropna(subset=existing_numeric_cols)  # drop rows with NaNs in key cols
after_rows = len(df_clean)   # rows after dropping
print(f"Dropped NaNs in {existing_numeric_cols}: {before_rows} → {after_rows}")

# ---------- 4) Handle duplicate rows ----------
total_duplicates = df_clean.duplicated().sum()  # count duplicate rows
print("Total duplicate rows:", total_duplicates)

if total_duplicates > 0:                 # if duplicates exist
    df_clean = df_clean.drop_duplicates() # remove duplicates
    print("✅ Duplicates removed. New shape:", df_clean.shape)
else:
    print("✅ No duplicates found!")

# ---------- 5) Ensure MP is numeric ----------
# Sometimes MP may be stored as text; we convert to numeric for computations
if "MP" in df_clean.columns:
    df_clean["MP"] = pd.to_numeric(df_clean["MP"], errors="coerce")  # invalid → NaN
    df_clean = df_clean.dropna(subset=["MP"])  # drop rows where MP could not be converted

print("\nDataFrame info after cleaning:")
df_clean.info()  # show final data types and non-null counts


📋 Missing values summary (non-zero only):


,Missing_Count,Missing_Percentage


Filtered MP > 0: 16512 → 16508 rows
Dropped NaNs in ['PTS', 'AST', 'TRB', 'FGA', 'FG', 'FTA', 'TOV']: 16508 → 16508
Total duplicate rows: 0
✅ No duplicates found!

DataFrame info after cleaning:
<class 'pandas.core.frame.DataFrame'>
Index: 16508 entries, 0 to 16511
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Player  16508 non-null  object 
 1   Tm      16508 non-null  object 
 2   Opp     16508 non-null  object 
 3   Res     16508 non-null  object 
 4   MP      16508 non-null  float64
 5   FG      16508 non-null  int64  
 6   FGA     16508 non-null  int64  
 7   FG%     16508 non-null  float64
 8   3P      16508 non-null  int64  
 9   3PA     16508 non-null  int64  
 10  3P%     16508 non-null  float64
 11  FT      16508 non-null  int64  
 12  FTA     16508 non-null  int64  
 13  FT%     16508 non-null  float64
 14  ORB     16508 non-null  int64  
 15  DRB     16508 non-null  int64  
 16  TRB     16508 non-null  int64 

In [4]:
# ============================================================
# CELL 4: FEATURE ENGINEERING (Leakage-safe)
# ============================================================
# Goal of this cell:
# - Create NEW features that help prediction
# - IMPORTANT: Since target is PTS, we must NOT use PTS in any feature
#   (otherwise the model "cheats" by seeing the answer inside X)

df_fe = df_clean.copy()  # create a working copy for feature engineering

print("🔧 Creating new features (Leakage-safe)")
print("=" * 60)

# ---- Feature 1: Assist-to-Turnover Ratio ----
# Why? Measures how efficient a player is as a playmaker (high AST, low TOV is good)
# +1 in denominator avoids division by zero
if {"AST", "TOV"}.issubset(df_fe.columns):
    df_fe["AST_TO_Ratio"] = df_fe["AST"] / (df_fe["TOV"] + 1)  # new column
    print("✅ Created feature: AST_TO_Ratio")

# ---- Feature 2: Rebound Activity ----
# Why? Another way to measure rebounding involvement; it combines offensive + defensive boards
if {"ORB", "DRB"}.issubset(df_fe.columns):
    df_fe["ReboundActivity"] = df_fe["ORB"] + df_fe["DRB"]  # ORB + DRB
    print("✅ Created feature: ReboundActivity")

# ---- Feature 3: Fouls per minute ----
# Why? Normalizes fouls by playing time; helps model understand aggressiveness/defense style
if {"PF", "MP"}.issubset(df_fe.columns):
    df_fe["PF_per_Min"] = df_fe["PF"] / (df_fe["MP"] + 1e-6)  # small epsilon to avoid /0
    print("✅ Created feature: PF_per_Min")

print("\nNew shape after feature engineering:", df_fe.shape)
df_fe.head()  # quick check


🔧 Creating new features (Leakage-safe)
✅ Created feature: AST_TO_Ratio
✅ Created feature: ReboundActivity
✅ Created feature: PF_per_Min

New shape after feature engineering: (16508, 28)


,Player,Tm,Opp,Res,MP,FG,FGA,FG%,3P,3PA,3P%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,GmSc,Data,AST_TO_Ratio,ReboundActivity,PF_per_Min
0,Jayson Tatum,BOS,NYK,W,30.30,14,18,0.778,8,11,0.727,1,2,0.500,0,4,4,10,1,1,1,1,37,38.1,2024-10-22,5.0,4,0.033003
1,Anthony Davis,LAL,MIN,W,37.58,11,23,0.478,1,3,0.333,13,15,0.867,3,13,16,4,1,3,1,1,36,34.0,2024-10-22,2.0,16,0.026610
2,Derrick White,BOS,NYK,W,26.63,8,13,0.615,6,10,0.600,2,2,1.000,0,3,3,4,1,0,0,1,24,22.4,2024-10-22,4.0,3,0.037552
3,Jrue Holiday,BOS,NYK,W,30.52,7,9,0.778,4,6,0.667,0,0,0.000,2,2,4,4,1,0,0,2,18,19.5,2024-10-22,4.0,4,0.065531
4,Miles McBride,NYK,BOS,L,25.85,8,10,0.800,4,5,0.800,2,3,0.667,0,0,0,2,0,0,1,1,22,17.8,2024-10-22,1.0,0,0.038685


In [5]:
# ============================================================
# CELL 5: SAVE CLEANED + FEATURE-ENGINEERED DATASET
# ============================================================
# Goal of this cell:
# - Save a CSV version of cleaned data for later tasks (clustering & classification)
# - This ensures we use the SAME cleaned version across the whole project

output_file = "cleaned_data.csv"            # output file name
df_fe.to_csv(output_file, index=False)      # save DataFrame to CSV without row index

print(f"✅ Saved dataset as {output_file}")
print("Saved dataset shape:", df_fe.shape)


✅ Saved dataset as cleaned_data.csv
Saved dataset shape: (16508, 28)


In [6]:
# ============================================================
# CELL 6: TRAIN/TEST SPLIT (Initial)
# ============================================================
# Goal of this cell:
# - Choose the target column (PTS)
# - Split X and y
# - Create train/test sets (80/20) for fair evaluation

target = "PTS"                   # regression target (what we want to predict)

# X = all columns except the target
X = df_fe.drop(columns=[target])  # remove PTS from input features
y = df_fe[target]                 # y is the target series (PTS values)

# Split the data into training and testing sets
# test_size=0.20 -> 20% for testing, 80% for training
# random_state -> reproducible split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Train samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])


Train samples: 13206
Test samples: 3302


In [7]:
# ============================================================
# CELL 7: BUILD CLEAN FEATURE SET (Remove leakage)
# ============================================================
# Goal of this cell:
# - Remove columns that can "give away" the target PTS
# - We remove shooting makes/attempts and percentages because they directly determine points
# - We also keep only numeric columns for now (simple baseline)

target = "PTS"  # make sure target is defined

# These columns strongly leak scoring information:
# Example: If we keep FG/3P/FT, the model can almost reconstruct PTS.
leak_cols = [
    "FG", "FGA", "FG%",
    "3P", "3PA", "3P%",
    "2P", "2PA", "2P%",
    "FT", "FTA", "FT%",
    "GmSc", "+/-",
    # safety: if these exist, remove them as well
    "ShootingEfficiency", "TrueShooting", "PTS_per_Min"
]

# Keep only leakage columns that actually exist in dataset
leak_cols = [c for c in leak_cols if c in df_fe.columns]

print("🚫 Leakage columns removed:", leak_cols)

# Select numeric columns only (ignore Player/Tm/Opp/Date because they are categorical)
numeric_cols = df_fe.select_dtypes(include=[np.number]).columns.tolist()

# Feature columns = numeric columns minus leakage columns minus target
feature_cols = [c for c in numeric_cols if c not in leak_cols + [target]]

print("✅ Number of usable features:", len(feature_cols))
print("✅ Feature list:", feature_cols)

# Build final X and y
X = df_fe[feature_cols]  # final input matrix
y = df_fe[target]        # final target vector

# Split again (same 80/20) so ALL models use the same clean feature set
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Final Train shape:", X_train.shape)
print("Final Test shape:", X_test.shape)


🚫 Leakage columns removed: ['FG', 'FGA', 'FG%', '3P', '3PA', '3P%', 'FT', 'FTA', 'FT%', 'GmSc']
✅ Number of usable features: 12
✅ Feature list: ['MP', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'AST_TO_Ratio', 'ReboundActivity', 'PF_per_Min']
Final Train shape: (13206, 12)
Final Test shape: (3302, 12)


In [8]:
# ============================================================
# CELL 8: LINEAR REGRESSION MODEL
# ============================================================
# Goal:
# - Train a baseline model (Linear Regression)
# - Evaluate using RMSE and R² on both train and test
from sklearn.linear_model import LinearRegression

# Create Linear Regression model object
lin_reg = LinearRegression()  # sklearn linear regression (fits a line/plane)

# Train/Fit the model on training data (learns coefficients)
lin_reg.fit(X_train, y_train)

# Predict PTS for training data
y_train_pred_lr = lin_reg.predict(X_train)

# Predict PTS for test data
y_test_pred_lr = lin_reg.predict(X_test)

# Compute Mean Squared Error (MSE)
mse_train_lr = mean_squared_error(y_train, y_train_pred_lr)
mse_test_lr  = mean_squared_error(y_test,  y_test_pred_lr)

# Convert MSE to RMSE (Root Mean Squared Error) -> same unit as PTS
rmse_train_lr = np.sqrt(mse_train_lr)
rmse_test_lr  = np.sqrt(mse_test_lr)

# Compute R² (explained variance score)
r2_train_lr = r2_score(y_train, y_train_pred_lr)
r2_test_lr  = r2_score(y_test,  y_test_pred_lr)

print("Linear Regression Results")
print("-" * 50)
print(f"Train RMSE: {rmse_train_lr:.3f} | Train R²: {r2_train_lr:.3f}")
print(f"Test  RMSE: {rmse_test_lr:.3f} | Test  R²: {r2_test_lr:.3f}")

# Show model intercept and coefficients (what the model learned)
print("\nIntercept:", lin_reg.intercept_)

coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": lin_reg.coef_
}).sort_values("coefficient", ascending=False)

display(coef_df.head(10))


Linear Regression Results
--------------------------------------------------
Train RMSE: 5.785 | Train R²: 0.576
Test  RMSE: 5.691 | Test  R²: 0.562

Intercept: -2.5507741156832164


,feature,coefficient
11,PF_per_Min,0.782795
4,AST,0.543645
0,MP,0.529121
7,TOV,0.406034
2,DRB,0.089248
10,ReboundActivity,0.028154
3,TRB,0.028154
5,STL,0.014858
1,ORB,-0.061094
6,BLK,-0.091646


In [9]:
# ============================================================
# CELL 9: RANDOM FOREST REGRESSOR
# ============================================================
# Goal:
# - Train a powerful non-linear model (Random Forest)
# - Random Forest = many decision trees averaged together
# - Usually performs well on complex relationships
from sklearn.ensemble import RandomForestRegressor  # Random Forest regression model

# Create Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=300,   # number of trees (more trees -> more stable, but slower)
    max_depth=None,     # None means trees can grow deep (can overfit, but RF reduces it)
    random_state=42,    # reproducibility
    n_jobs=-1           # use all CPU cores for faster training
)

# Train the model
rf_model.fit(X_train, y_train)

# Predictions on training data
y_train_pred_rf = rf_model.predict(X_train)

# Predictions on testing data
y_test_pred_rf  = rf_model.predict(X_test)

# Evaluate MSE
mse_train_rf = mean_squared_error(y_train, y_train_pred_rf)
mse_test_rf  = mean_squared_error(y_test,  y_test_pred_rf)

# Evaluate RMSE
rmse_train_rf = np.sqrt(mse_train_rf)
rmse_test_rf  = np.sqrt(mse_test_rf)

# Evaluate R²
r2_train_rf = r2_score(y_train, y_train_pred_rf)
r2_test_rf  = r2_score(y_test,  y_test_pred_rf)

print("Random Forest Results")
print("-" * 50)
print(f"Train RMSE: {rmse_train_rf:.3f} | Train R²: {r2_train_rf:.3f}")
print(f"Test  RMSE: {rmse_test_rf:.3f} | Test  R²: {r2_test_rf:.3f}")

# Feature importance = how much each feature contributed to decision splits
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance_df.head(10))


Random Forest Results
--------------------------------------------------
Train RMSE: 2.186 | Train R²: 0.939
Test  RMSE: 5.760 | Test  R²: 0.551


,feature,importance
0,MP,0.668665
11,PF_per_Min,0.077192
9,AST_TO_Ratio,0.036930
7,TOV,0.032547
2,DRB,0.031281
4,AST,0.031243
5,STL,0.026264
1,ORB,0.023998
3,TRB,0.021095
10,ReboundActivity,0.021012


In [10]:
# ============================================================
# CELL 10: SUPPORT VECTOR REGRESSION (SVR)
# ============================================================
# Goal:
# - Train SVR with RBF kernel (non-linear)
# - SVR requires scaling because it is distance-based
from sklearn.svm import SVR
# Create the scaler object (StandardScaler)
scaler = StandardScaler()

# Fit scaler on TRAIN features only (avoid leakage), then transform train
X_train_scaled = scaler.fit_transform(X_train)

# Transform test using the same scaler parameters learned from train
X_test_scaled  = scaler.transform(X_test)

# Create SVR model
svr_model = SVR(
    kernel="rbf",  # RBF = radial basis function, captures non-linear patterns
    C=10.0,        # regularization (bigger C => more flexible model)
    epsilon=0.1    # tolerance tube (ignores small errors within epsilon)
)

# Train SVR using scaled training data
svr_model.fit(X_train_scaled, y_train)

# Predict on scaled training data
y_train_pred_svr = svr_model.predict(X_train_scaled)

# Predict on scaled testing data
y_test_pred_svr  = svr_model.predict(X_test_scaled)

# Evaluate MSE
mse_train_svr = mean_squared_error(y_train, y_train_pred_svr)
mse_test_svr  = mean_squared_error(y_test,  y_test_pred_svr)

# Evaluate RMSE
rmse_train_svr = np.sqrt(mse_train_svr)
rmse_test_svr  = np.sqrt(mse_test_svr)

# Evaluate R²
r2_train_svr = r2_score(y_train, y_train_pred_svr)
r2_test_svr  = r2_score(y_test,  y_test_pred_svr)

print("SVR (RBF Kernel) Results")
print("-" * 50)
print(f"Train RMSE: {rmse_train_svr:.3f} | Train R²: {r2_train_svr:.3f}")
print(f"Test  RMSE: {rmse_test_svr:.3f} | Test  R²: {r2_test_svr:.3f}")


SVR (RBF Kernel) Results
--------------------------------------------------
Train RMSE: 5.489 | Train R²: 0.618
Test  RMSE: 5.650 | Test  R²: 0.568


In [11]:
# ============================================================
# CELL 11: MODEL COMPARISON SUMMARY
# ============================================================
# Goal:
# - Put all model results into one table for easy comparison
# - Sort by Test RMSE (lower is better)
# - This table is what you can paste into your report

comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "SVR (RBF)"],
    
    # RMSE values: lower means predictions are closer to real PTS
    "Train RMSE": [rmse_train_lr, rmse_train_rf, rmse_train_svr],
    "Test RMSE":  [rmse_test_lr,  rmse_test_rf,  rmse_test_svr],
    
    # R² values: higher means model explains more variance in PTS
    "Train R²":   [r2_train_lr, r2_train_rf, r2_train_svr],
    "Test R²":    [r2_test_lr,  r2_test_rf,  r2_test_svr]
})

# Sort models by best generalization (lowest test RMSE)
comparison_sorted = comparison.sort_values("Test RMSE")

print("📊 Model Comparison (sorted by best Test RMSE):")
display(comparison_sorted)


📊 Model Comparison (sorted by best Test RMSE):


,Model,Train RMSE,Test RMSE,Train R²,Test R²
2,SVR (RBF),5.489464,5.649960,0.618343,0.567953
0,Linear Regression,5.784915,5.690506,0.576155,0.561730
1,Random Forest,2.186140,5.759565,0.939470,0.551027


Training set: 13,206 samples (80.0%)
Test set:     3,302 samples (20.0%)


,Player,Tm,Opp,Res,MP,FG,FGA,FG%,3P,3PA,3P%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,GmSc,Data,ShootingEfficiency,TrueShooting,AST_TO_Ratio
15899,Kristaps Porziņģis,BOS,CLE,W,34.65,7,20,0.350,1,6,0.167,4,4,1.00,1,6,7,2,2,2,2,4,11.5,2025-02-04,0.904762,0.417399,0.666667
704,Julian Strawther,DEN,LAC,L,16.20,3,5,0.600,2,2,1.000,0,0,0.00,1,1,2,3,2,0,1,5,7.8,2024-10-26,1.333333,0.666667,1.500000
12111,Collin Sexton,UTA,PHO,L,28.88,8,17,0.471,1,6,0.167,3,4,0.75,2,2,4,4,0,0,2,2,12.9,2025-01-11,1.111111,0.506073,1.333333
2417,Haywood Highsmith,MIA,PHO,L,30.60,7,8,0.875,2,3,0.667,3,3,1.00,2,5,7,0,2,0,0,3,19.9,2024-11-06,2.111111,0.920543,0.000000
2655,Jett Howard,ORL,IND,L,3.08,0,3,0.000,0,2,0.000,0,0,0.00,0,0,0,0,0,0,1,1,-3.5,2024-11-06,0.000000,0.000000,0.000000
